<a href="https://colab.research.google.com/github/LaraDondossola/Classificacao-eventos-climaticos/blob/main/Notebooks/Interface_eventos_clim%C3%A1ticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit joblib pandas scikit-learn
!npm install -g localtunnel
!pip install -q pyngrok streamlit

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [ ]:
%%writefile app.py
import streamlit as st
import joblib
import pandas as pd
import numpy as np

# Configuração da página
st.set_page_config(
    page_title="Painel de Impacto de Eventos Climáticos",
    page_icon="🌊",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Estilização CSS customizada
st.markdown("""
<style>
    .main-title {
        font-size: 2.6rem;
        font-weight: 800;
        color: #1E293B;
        margin-bottom: 0.3rem;
    }
    .sub-title {
        font-size: 1.2rem;
        color: #475569;
        margin-bottom: 1.8rem;
    }
    .metric-card {
        background-color: #F8FAFC;
        border: 1px solid #CBD5E1;
        border-radius: 14px;
        padding: 24px;
        text-align: center;
        box-shadow: 0 4px 10px -2px rgba(0, 0, 0, 0.08);
    }
    .metric-title {
        font-size: 1.15rem;
        font-weight: 700;
        color: #334155;
        text-transform: uppercase;
        letter-spacing: 0.05em;
    }
    .metric-value-knn {
        font-size: 2.8rem;
        font-weight: 800;
        color: #D97706;
    }
    .metric-value-rf {
        font-size: 2.8rem;
        font-weight: 800;
        color: #059669;
    }
    .metric-value-diff {
        font-size: 2.8rem;
        font-weight: 800;
        color: #2563EB;
    }
    .metric-subtitle {
        color: #475569;
        font-size: 1.05rem;
        margin-top: 6px;
        font-weight: 500;
    }
    .metric-pct {
        font-weight: 700;
        font-size: 1.25rem;
        color: #0F172A;
    }
    .risk-badge {
        display: inline-block;
        padding: 8px 18px;
        border-radius: 20px;
        font-weight: 800;
        font-size: 1.1rem;
        margin-top: 10px;
    }
    .risk-baixo { background-color: #DCFCE7; color: #166534; }
    .risk-medio { background-color: #FEF3C7; color: #92400E; }
    .risk-alto { background-color: #FEE2E2; color: #991B1B; }

    .historic-cuts-title {
        color: #475569;
        font-size: 1rem;
        font-weight: 600;
        margin-top: 6px;
    }
    .historic-cuts-val {
        font-size: 0.95rem;
        color: #334155;
        font-weight: 600;
        margin-top: 4px;
    }
</style>
""", unsafe_allow_html=True)

# ==============================================================================
# CARREGAMENTO DOS ARTEFATOS
# ==============================================================================
@st.cache_resource
def carregar_recursos():
    preprocessor = joblib.load('preprocessor.pkl')
    knn = joblib.load('modelo_knn_regressor.pkl')
    rf = joblib.load('modelo_random_forest.pkl')
    return preprocessor, knn, rf

try:
    preprocessor, modelo_knn, modelo_rf = carregar_recursos()
except Exception as e:
    st.error(f"⚠️ Erro ao carregar os ficheiros .pkl: {e}")
    st.stop()

def classificar_risco(percentual):
    if percentual < 3.6034:
        return "Baixo", "risk-baixo"
    elif percentual <= 50.1603:
        return "Médio", "risk-medio"
    else:
        return "Alto", "risk-alto"

# ==============================================================================
# CABEÇALHO
# ==============================================================================
st.markdown('<p class="main-title">🌊 Dashboard de Estimativa de Impacto Climático</p>', unsafe_allow_html=True)
st.markdown('<p class="sub-title">Preveja e compare a população afetada por desastres utilizando o KNN Regressor e o Random Forest Regressor.</p>', unsafe_allow_html=True)

st.divider()

tipos_eventos_cobrade = [
    "Enxurradas",
    "Inundações",
    "Alagamentos",
    "Tempestade Local/Convectiva - Chuvas Intensas",
    "Tempestade Local/Convectiva - Vendaval",
    "Tempestade Local/Convectiva - Granizo",
    "Tempestade Local/Convectiva - Raios",
    "Deslizamentos",
    "Corrida de Massa / Fluxo de Lama",
    "Tornado / Ciclone",
    "Estiagem e Seca",
    "Geada / Onda de Frio",
    "Onda de Calor",
    "Outros"
]

# ==============================================================================
# FORMULÁRIO DE ENTRADA
# ==============================================================================
with st.form("form_evento"):
    col_left, col_right = st.columns([1, 1], gap="large")

    with col_left:
        st.subheader("📍 1. Localização e Evento")
        uf = st.selectbox(
            "Unidade Federativa (UF)",
            ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"],
            index=23
        )
        populacao = st.number_input("População Total do Município", min_value=1, value=50000, step=1000)
        tipo_evento = st.selectbox(
            "Tipo do Evento Climático (COBRADE)",
            tipos_eventos_cobrade,
            index=3
        )

    with col_right:
        st.subheader("🏠 2. Danos Materiais Informados")
        uh_danificadas = st.number_input("Unidades Habitacionais Danificadas", min_value=0, value=10, step=5)
        uh_destruidas = st.number_input("Unidades Habitacionais Destruídas", min_value=0, value=0, step=1)
        obras_infra = st.number_input("Obras de Infraestrutura Pública Destruídas/Danificadas", min_value=0, value=0, step=1)

    st.write("")
    btn_prever = st.form_submit_button("🚀 Gerar Estimativas Comparativas", use_container_width=True)

# ==============================================================================
# PROCESSAMENTO E EXIBIÇÃO DE RESULTADOS
# ==============================================================================
if btn_prever:
    try:
        colunas_cat = []
        colunas_num = []

        if hasattr(preprocessor, 'transformers_'):
            for name, trans, cols in preprocessor.transformers_:
                if name == 'cat':
                    colunas_cat.extend(cols)
                elif name == 'num':
                    colunas_num.extend(cols)
                elif name != 'remainder' and isinstance(cols, list):
                    if 'onehot' in name.lower() or 'encoder' in name.lower():
                        colunas_cat.extend(cols)
                    else:
                        colunas_num.extend(cols)

        colunas_todas = getattr(preprocessor, 'feature_names_in_', list(set(colunas_cat + colunas_num)))

        dados_completos = {}
        for col in colunas_todas:
            if col in colunas_cat or col in ['UF', 'Tipo_Evento']:
                dados_completos[col] = "Não Informado"
            else:
                dados_completos[col] = 0.0

        df_input = pd.DataFrame([dados_completos])

        # Tratar string do Tipo_Evento para bater exatamente com o split do script de treino
        tipo_evento_limpo = tipo_evento.split(' - ')[0].strip() if ' - ' in tipo_evento else tipo_evento

        if 'UF' in df_input.columns:
            df_input['UF'] = uf
        if 'Tipo_Evento' in df_input.columns:
            df_input['Tipo_Evento'] = tipo_evento_limpo
        if 'População' in df_input.columns:
            df_input['População'] = float(populacao)
        if 'Log_Populacao' in df_input.columns:
            df_input['Log_Populacao'] = float(np.log1p(populacao))

        # Atribuição dos danos habitacionais e de infraestrutura
        for col in df_input.columns:
            if 'Habitacionais' in col and 'Danificadas' in col and 'Valor' not in col:
                df_input[col] = float(uh_danificadas)
            elif 'Habitacionais' in col and 'Destruídas' in col and 'Valor' not in col:
                df_input[col] = float(uh_destruidas)
            elif 'infraestrutura' in col.lower() and 'Danificadas' in col and 'Valor' not in col:
                df_input[col] = float(obras_infra)

        # Recálculo das Colunas Agregadas (Feature Engineering)
        cols_danificadas = [c for c in df_input.columns if 'Danificadas' in c and 'Total' not in c and c != 'População_Afetada_Total']
        cols_destruidas = [c for c in df_input.columns if 'Destruídas' in c and 'Total' not in c and c != 'População_Afetada_Total']

        if 'Total_Estruturas_Danificadas' in df_input.columns:
            df_input['Total_Estruturas_Danificadas'] = df_input[cols_danificadas].sum(axis=1) if cols_danificadas else float(uh_danificadas)

        if 'Total_Estruturas_Destruidas' in df_input.columns:
            df_input['Total_Estruturas_Destruidas'] = df_input[cols_destruidas].sum(axis=1) if cols_destruidas else float(uh_destruidas + obras_infra)

        # Garantir preenchimento dos tipos corretos
        for col in df_input.columns:
            if col not in colunas_cat and col not in ['UF', 'Tipo_Evento']:
                df_input[col] = pd.to_numeric(df_input[col], errors='coerce').fillna(0.0)

        # Transformação e Predição
        X_novo_prep = preprocessor.transform(df_input)

        pred_knn_log = modelo_knn.predict(X_novo_prep)
        pred_rf_log = modelo_rf.predict(X_novo_prep)

        pred_knn_pessoas = max(0, int(round(np.expm1(pred_knn_log)[0])))
        pred_rf_pessoas = max(0, int(round(np.expm1(pred_rf_log)[0])))
        diferenca = abs(pred_knn_pessoas - pred_rf_pessoas)

        pct_knn = (pred_knn_pessoas / populacao) * 100
        pct_rf = (pred_rf_pessoas / populacao) * 100

        risco_knn, classe_css_knn = classificar_risco(pct_knn)
        risco_rf, classe_css_rf = classificar_risco(pct_rf)

        # Resultados em tela
        st.write("")
        st.subheader("📊 Previsão de População Afetada e Nível de Risco")

        res_col1, res_col2, res_col3 = st.columns(3)

        with res_col1:
            st.markdown(f"""
            <div class="metric-card">
                <div class="metric-title">KNN Regressor</div>
                <div class="metric-value-knn">{pred_knn_pessoas:,}</div>
                <div class="metric-subtitle">Pessoas estimadas</div>
                <hr style="margin: 14px 0; border: 0; border-top: 1px solid #CBD5E1;">
                <div class="metric-pct">{pct_knn:.2f}% da população</div>
                <div class="risk-badge {classe_css_knn}">Risco {risco_knn}</div>
            </div>
            """.replace(",", "."), unsafe_allow_html=True)

        with res_col2:
            st.markdown(f"""
            <div class="metric-card">
                <div class="metric-title">Random Forest Regressor</div>
                <div class="metric-value-rf">{pred_rf_pessoas:,}</div>
                <div class="metric-subtitle">Pessoas estimadas</div>
                <hr style="margin: 14px 0; border: 0; border-top: 1px solid #CBD5E1;">
                <div class="metric-pct">{pct_rf:.2f}% da população</div>
                <div class="risk-badge {classe_css_rf}">Risco {risco_rf}</div>
            </div>
            """.replace(",", "."), unsafe_allow_html=True)

        with res_col3:
            st.markdown(f"""
            <div class="metric-card">
                <div class="metric-title">Diferença Absoluta</div>
                <div class="metric-value-diff">{diferenca:,}</div>
                <div class="metric-subtitle">Variação entre os modelos</div>
                <hr style="margin: 14px 0; border: 0; border-top: 1px solid #CBD5E1;">
                <div class="historic-cuts-title">Cortes históricos:</div>
                <div class="historic-cuts-val">
                    Baixo (&lt;3,6%) | Médio (3,6%-50,2%) | Alto (&gt;50,2%)
                </div>
            </div>
            """.replace(",", "."), unsafe_allow_html=True)

    except Exception as e:
        st.error(f"⚠️ Ocorreu um erro durante a predição: {e}")

Overwriting app.py


In [ ]:
from pyngrok import ngrok
import os

# 1. Configurar o seu Authtoken (substitua com o token do site)
ngrok.set_auth_token("3Eu0uGhV8Sw9rvvCGXlcwhNKIfj_81kRpesKtmutVx3YxxhhF")

# 2. Encerrar conexões anteriores
ngrok.kill()

# 3. Criar o túnel na porta 8501
public_url = ngrok.connect(8501)
print(f"🔗 Acesse sua aplicação sem erros aqui: {public_url}")

# 4. Rodar o Streamlit
!streamlit run app.py --server.port 8501

🔗 Acesse sua aplicação sem erros aqui: NgrokTunnel: "https://slideshow-litter-outspoken.ngrok-free.dev" -> "http://localhost:8501"


2026-09-22 18:24:28.653 Port 8501 is not available
